In [1]:
import os
import numpy as np
from skimage import io, transform
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras import backend as K

# Set your paths to the dataset here.
IMAGE_PATH = '/Users/blood-vessel-segmentation/train/images'
LABEL_PATH = '/Users/blood-vessel-segmentation/train/labels'

## Dataset and Model Performance Summary

The dataset used in this notebook can be downloaded from the official [Kaggle competition](https://www.kaggle.com/competitions/blood-vessel-segmentation/data).

This notebook evaluates the performance of a **Random Forest model** on the **blood vessel segmentation task using 3D Hierarchical Phase-Contrast Tomography data**. The results demonstrate that Random Forest is **not well-suited** for this type of segmentation, as it fails to capture spatial dependencies and class imbalances effectively.

The model achieved a **Dice coefficient of 0.0000**, indicating that it predominantly predicted the background and failed to segment vessels correctly. Alternative approaches, such as **deep learning-based methods (e.g., U-Net)**, are better suited for this dataset.

In [2]:
# Custom function to preprocess images
def preprocess_image(image, target_resolution=(128, 128), max_pixel_value=255.0):
    # Normalize images by max pixel value to keep the range [0, 1]
    img = transform.resize(image, target_resolution, mode='constant', anti_aliasing=True)
    img = img.astype('float32') / max_pixel_value
    return img

# Custom function to preprocess masks
def preprocess_mask(mask, target_resolution=(128, 128)):
    mask = transform.resize(mask, target_resolution, order=0, mode='constant', preserve_range=True)
    mask = mask.astype('float32') / 255.0
    mask = (mask > 0.5).astype('uint8')  # Binarize the mask
    return mask

# Load and preprocess images and masks
image_files = sorted([f for f in os.listdir(IMAGE_PATH) if f.endswith('.tif')])
label_files = sorted([f for f in os.listdir(LABEL_PATH) if f.endswith('.tif')])

In [3]:
dataset_images = np.array([preprocess_image(io.imread(os.path.join(IMAGE_PATH, file)), target_resolution=(128, 128)) for file in image_files])
dataset_labels = np.array([preprocess_mask(io.imread(os.path.join(LABEL_PATH, file)), target_resolution=(128, 128)) for file in label_files])

# Reshape data to have pixels as individual samples
num_samples, height, width = dataset_images.shape
X = dataset_images.reshape(-1, 1)  # Each pixel as a feature
y = dataset_labels.reshape(-1)  # Flatten mask labels accordingly

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_val)

In [6]:
# Dice coefficient calculation
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

dice_score = dice_coefficient(y_val, y_pred)
print(f'Random Forest Dice Coefficient: {dice_score:.4f}')

Random Forest Dice Coefficient: 0.0000
